# RAG Dataset Analysis

Quick analysis of questions, answers, and prediction errors.

## 1. Setup

In [ ]:
import pandas as pd
import json
import re
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
# Load data
TRAIN_PATH = "/content/drive/MyDrive/Text_Retrieval_and_Search_Engines/rag_train.csv"
TEST_PATH = "/content/drive/MyDrive/Text_Retrieval_and_Search_Engines/rag_test.csv"

# Optional: Load your predictions for error analysis
PREDICTIONS_PATH = "/content/submission_v58_smart_retry.csv"  # Change as needed

df_train = pd.read_csv(TRAIN_PATH, converters={"answers": json.loads})
df_test = pd.read_csv(TEST_PATH, converters={"answers": json.loads})

print(f"Train: {len(df_train)} questions")
print(f"Test: {len(df_test)} questions")

## 2. Question Type Analysis

In [ ]:
def classify_question_type(question):
    """Classify question by its starting word."""
    q = question.lower().strip()
    
    if q.startswith('who '):
        return 'who'
    elif q.startswith('what '):
        return 'what'
    elif q.startswith('where '):
        return 'where'
    elif q.startswith('when '):
        return 'when'
    elif q.startswith('which '):
        return 'which'
    elif q.startswith('how '):
        return 'how'
    elif q.startswith('why '):
        return 'why'
    elif q.startswith(('is ', 'are ', 'was ', 'were ')):
        return 'yes/no'
    elif q.startswith(('name ', 'give ', 'list ')):
        return 'list'
    else:
        return 'other'

In [ ]:
# Classify all questions
df_train['q_type'] = df_train['question'].apply(classify_question_type)
df_test['q_type'] = df_test['question'].apply(classify_question_type)

print("=== TRAIN Question Types ===")
print(df_train['q_type'].value_counts())
print(f"\nPercentages:")
print((df_train['q_type'].value_counts() / len(df_train) * 100).round(1))

In [ ]:
print("\n=== TEST Question Types ===")
print(df_test['q_type'].value_counts())

## 3. Subject/Topic Analysis

In [ ]:
def extract_subject_category(question):
    """Categorize question by subject matter."""
    q = question.lower()
    
    # Geography
    if any(w in q for w in ['country', 'capital', 'city', 'continent', 'river', 'mountain', 'ocean', 'island']):
        return 'geography'
    
    # Currency/Economics
    if any(w in q for w in ['currency', 'money', 'dollar', 'euro', 'franc', 'peso', 'pound']):
        return 'currency'
    
    # People/Biography
    if any(w in q for w in ['born', 'died', 'married', 'founded', 'invented', 'discovered', 'wrote', 'directed']):
        return 'biography'
    
    # Entertainment
    if any(w in q for w in ['movie', 'film', 'actor', 'actress', 'played', 'character', 'song', 'album', 'band']):
        return 'entertainment'
    
    # Sports
    if any(w in q for w in ['team', 'player', 'championship', 'olympics', 'world cup', 'sport', 'league']):
        return 'sports'
    
    # Science
    if any(w in q for w in ['element', 'planet', 'species', 'chemical', 'scientific', 'theory']):
        return 'science'
    
    # History
    if any(w in q for w in ['war', 'battle', 'president', 'king', 'queen', 'empire', 'dynasty']):
        return 'history'
    
    # Politics/Government
    if any(w in q for w in ['government', 'minister', 'president', 'election', 'political', 'party']):
        return 'politics'
    
    # Literature
    if any(w in q for w in ['book', 'novel', 'author', 'wrote', 'poetry', 'poem']):
        return 'literature'
    
    return 'other'

In [ ]:
df_train['subject'] = df_train['question'].apply(extract_subject_category)
df_test['subject'] = df_test['question'].apply(extract_subject_category)

print("=== TRAIN Subject Categories ===")
print(df_train['subject'].value_counts())
print(f"\nPercentages:")
print((df_train['subject'].value_counts() / len(df_train) * 100).round(1))

In [ ]:
print("\n=== TEST Subject Categories ===")
print(df_test['subject'].value_counts())

## 4. Answer Length Analysis

In [ ]:
# Analyze gold answer lengths (train only)
df_train['answer_lengths'] = df_train['answers'].apply(lambda x: [len(a.split()) for a in x])
df_train['min_answer_len'] = df_train['answer_lengths'].apply(min)
df_train['max_answer_len'] = df_train['answer_lengths'].apply(max)

print("=== Answer Length Distribution (words) ===")
print(df_train['min_answer_len'].describe())

In [ ]:
# Distribution
len_counts = df_train['min_answer_len'].value_counts().sort_index()
print("\nAnswer length frequency:")
for length, count in len_counts.head(10).items():
    print(f"  {length} word(s): {count} ({100*count/len(df_train):.1f}%)")

## 5. Cross-Analysis: Question Type × Subject

In [ ]:
cross_tab = pd.crosstab(df_train['q_type'], df_train['subject'])
print("=== Question Type × Subject (Train) ===")
print(cross_tab)

## 6. Sample Questions by Category

In [ ]:
def show_samples(df, category_col, category_val, n=5):
    """Show sample questions for a category."""
    samples = df[df[category_col] == category_val].sample(min(n, len(df[df[category_col] == category_val])))
    print(f"\n=== Sample '{category_val}' Questions ===")
    for _, row in samples.iterrows():
        print(f"Q: {row['question']}")
        if 'answers' in row:
            print(f"A: {row['answers']}")
        print()

In [ ]:
# Show samples for each subject
for subject in ['currency', 'entertainment', 'geography', 'biography']:
    show_samples(df_train, 'subject', subject, n=3)

## 7. Error Analysis (if predictions available)

In [ ]:
# Load predictions if available
try:
    df_pred = pd.read_csv(PREDICTIONS_PATH)
    df_pred['prediction'] = df_pred['prediction'].apply(lambda x: json.loads(x)[0])
    print(f"Loaded {len(df_pred)} predictions")
    HAS_PREDICTIONS = True
except:
    print("No predictions file found - skipping error analysis")
    HAS_PREDICTIONS = False

In [ ]:
if HAS_PREDICTIONS:
    # Merge with train for evaluation
    df_eval = df_train.merge(df_pred, on='id', how='inner')
    print(f"Matched {len(df_eval)} predictions with gold answers")
    
    # Calculate F1 for each
    import string
    from collections import Counter
    
    def normalize_answer(s):
        s = s.lower()
        s = re.sub(r'\b(a|an|the)\b', ' ', s)
        s = ''.join(c for c in s if c not in set(string.punctuation))
        return ' '.join(s.split())
    
    def f1_score(pred, gold):
        pred_tokens = normalize_answer(pred).split()
        gold_tokens = normalize_answer(gold).split()
        common = Counter(pred_tokens) & Counter(gold_tokens)
        num_same = sum(common.values())
        if len(pred_tokens) == 0 or len(gold_tokens) == 0:
            return int(pred_tokens == gold_tokens)
        if num_same == 0:
            return 0
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gt_tokens)
        return (2 * precision * recall) / (precision + recall)
    
    def max_f1(pred, golds):
        return max(f1_score(pred, g) for g in golds)
    
    df_eval['f1'] = df_eval.apply(lambda r: max_f1(r['prediction'], r['answers']), axis=1)
    
    print(f"\nOverall F1: {df_eval['f1'].mean()*100:.2f}%")

In [ ]:
if HAS_PREDICTIONS:
    # F1 by question type
    print("\n=== F1 by Question Type ===")
    for q_type in df_eval['q_type'].unique():
        subset = df_eval[df_eval['q_type'] == q_type]
        avg_f1 = subset['f1'].mean() * 100
        print(f"  {q_type:10s}: {avg_f1:5.2f}% (n={len(subset)})")

In [ ]:
if HAS_PREDICTIONS:
    # F1 by subject
    print("\n=== F1 by Subject ===")
    for subject in df_eval['subject'].unique():
        subset = df_eval[df_eval['subject'] == subject]
        avg_f1 = subset['f1'].mean() * 100
        print(f"  {subject:15s}: {avg_f1:5.2f}% (n={len(subset)})")

In [ ]:
if HAS_PREDICTIONS:
    # Worst performing categories
    print("\n=== Failure Analysis (F1 = 0) ===")
    failures = df_eval[df_eval['f1'] == 0]
    print(f"Total failures: {len(failures)} ({100*len(failures)/len(df_eval):.1f}%)")
    
    print("\nFailures by question type:")
    print(failures['q_type'].value_counts())
    
    print("\nFailures by subject:")
    print(failures['subject'].value_counts())

In [ ]:
if HAS_PREDICTIONS:
    # Sample failures
    print("\n=== Sample Failures ===")
    for _, row in failures.sample(min(10, len(failures))).iterrows():
        print(f"Q: {row['question']}")
        print(f"Pred: {row['prediction']}")
        print(f"Gold: {row['answers']}")
        print(f"Type: {row['q_type']} | Subject: {row['subject']}")
        print()

## 8. Specific Pattern Analysis

In [ ]:
# Currency questions deep dive
currency_qs = df_train[df_train['subject'] == 'currency']
print(f"=== Currency Questions ({len(currency_qs)}) ===")
print("\nSample:")
for _, row in currency_qs.head(5).iterrows():
    print(f"Q: {row['question']}")
    print(f"A: {row['answers']}\n")

In [ ]:
# "Who played" questions
played_qs = df_train[df_train['question'].str.lower().str.contains('who played|who plays')]
print(f"=== 'Who Played' Questions ({len(played_qs)}) ===")
print("\nSample:")
for _, row in played_qs.head(5).iterrows():
    print(f"Q: {row['question']}")
    print(f"A: {row['answers']}\n")

## 9. Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTrain: {len(df_train)} questions")
print(f"Test:  {len(df_test)} questions")

print(f"\n--- Question Types (Train) ---")
for qt, count in df_train['q_type'].value_counts().items():
    print(f"  {qt:10s}: {count:4d} ({100*count/len(df_train):5.1f}%)")

print(f"\n--- Subjects (Train) ---")
for subj, count in df_train['subject'].value_counts().items():
    print(f"  {subj:15s}: {count:4d} ({100*count/len(df_train):5.1f}%)")

print(f"\n--- Answer Length (Train) ---")
print(f"  Avg: {df_train['min_answer_len'].mean():.1f} words")
print(f"  Max: {df_train['min_answer_len'].max()} words")
print(f"  1-word answers: {(df_train['min_answer_len'] == 1).sum()} ({100*(df_train['min_answer_len'] == 1).sum()/len(df_train):.1f}%)")
print(f"  2-word answers: {(df_train['min_answer_len'] == 2).sum()} ({100*(df_train['min_answer_len'] == 2).sum()/len(df_train):.1f}%)")

if HAS_PREDICTIONS:
    print(f"\n--- Model Performance ---")
    print(f"  Overall F1: {df_eval['f1'].mean()*100:.2f}%")
    print(f"  Perfect (F1=1): {(df_eval['f1'] == 1).sum()}")
    print(f"  Failures (F1=0): {(df_eval['f1'] == 0).sum()}")

print("\n" + "=" * 60)